# Taxanomy Creation


In [ ]:
'''the two `os.environ` lines are set before importing
sklearn to prevent an OpenMP library conflict that was crashing the kernel on this
Mac/conda setup. `OMP_NUM_THREADS=1` disables the threading that triggered it.'''

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
embeddings = np.load("data/embeddings.npy")  
print(embeddings.shape)

In [ ]:
import pandas as pd

In [ ]:
google_play = pd.read_csv('data/google_play_reviews.csv')
google_play = google_play.drop_duplicates()
google_play

In [ ]:
app_store = pd.read_csv('data/app_store_reviews.csv')
app_store = app_store.drop_duplicates()
app_store

### Final cleaning 

After combining, I drop any rows with no review text (nothing to analyze) and
normalize the `date` column to a real datetime so it's sortable for later
version/time work. Apple rows have no date (the RSS feed doesn't provide one),
so those stay null by design — the time-based analysis leans on Google Play,
which attaches both a date and an app version to every review.

In [ ]:
result = pd.concat([app_store, google_play], ignore_index=True)
print(result)
print(len(result)) 
result.to_csv("data/combined_reviews.csv", index=False)

In [ ]:
combined = pd.read_csv('data/combined_reviews.csv')
combined = combined.dropna(subset=['text'])
combined["date"] = pd.to_datetime(combined["date"])
combined

## Theme extraction: embedding the reviews

### Sentence Transformer 
A sentence transformer converts each review into a vector — a list of ~384 numbers
that represents the text's *meaning*. The defining property: texts with similar
meaning land close together in that number-space, even when they share no words.
So "the ranking system is confusing" and "I can't figure out how the rating works"
end up near each other despite using completely different vocabulary. Once every
review is a vector, "find the themes" becomes "find groups of nearby vectors" — a
problem clustering can solve.

### Why this method is a good fit here
- It captures meaning, not just keywords. Reviewers describe the same issue
  countless ways — typos, slang, abbreviations, different phrasings. A keyword or
  word-count approach would file "invite 4 friends," "forced to recommend it," and
  "harass others into joining" as three unrelated things; embeddings recognize them
  as one theme. For messy user-generated text, that semantic grouping is the whole
  point.
- The themes aren't known in advance. My first research question is literally
  "what do users talk about?" Embeddings let themes emerge from the data instead
  of forcing me to pre-specify a keyword list — which would bias the result toward
  what I already expected to find.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
reviews = combined["text"].tolist()
embeddings = model.encode(reviews, show_progress_bar=True)
print(embeddings.shape) 

In [ ]:
import numpy as np
np.save("data/embeddings.npy", embeddings)

## Clustering the embeddings into themes

### What KMeans does
KMeans groups the 825 review-vectors into *k* clusters by repeating two steps until
they stabilize: assign each point to its nearest cluster center, then move each
center to the average of its assigned points. When the centers stop moving, every
review belongs to the cluster it's closest to. The themes are then read off by
inspecting what's in each cluster.

### Why KMeans (and why not the alternatives)
- **KMeans vs. BERTopic / HDBSCAN:** I considered BERTopic, but it's built for a larger
  dataset and its HDBSCAN clustering strands uncertain points in an "outlier"
  bucket — on a small set like mine, that can leave a large share of reviews
  unclustered. KMeans assigns *every* review to a cluster, which suits 825 reviews
  far better.
- It's transparent and reproducible. Two understandable steps (embed, then
  cluster), and with a fixed `random_state` the results are stable run to run. For
  a portfolio project I need to *explain and defend* every step, and KMeans is
  simple enough to do that fully.

In [ ]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=3, random_state=42, n_init=5)
km.fit(embeddings)
print("survived:", km.inertia_)

In [ ]:
import matplotlib.pyplot as plt

inertias = []
K_range = range(2, 13) 

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(embeddings)
    inertias.append(km.inertia_)

plt.plot(K_range, inertias, marker="o")
plt.xlabel("k (number of clusters)")
plt.ylabel("inertia")
plt.title("Elbow method")
plt.show()

### How I chose k

KMeans needs *k* up front, so I fit across a range of *k* and plotted the inertia (total distance from points to their cluster centers) to see where adding clusters stopped helping. The drop-off was gradual rather than sharp, so I treated it as a range and picked the *k* whose clusters read as the most coherent themes (k=8), checking that sizes were reasonably balanced rather than one dominant catch-all.

In [ ]:
k = 8  

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
combined["cluster"] = kmeans.fit_predict(embeddings)


print(combined["cluster"].value_counts().sort_index())

In [ ]:
for c in range(8):
    subset = combined[combined["cluster"] == c]
    print(f"\n{'='*60}\nCLUSTER {c}  (n={len(subset)})\n{'='*60}")
    for s in subset["text"].head(8):
        print(" •", str(s)[:160])

### Results and Limitations

Some clusters were clean (map/navigation, the forced invite wall, positive/diary use), but others were muddy — the same complaints, especially the invite wall, showed up across multiple clusters. This exposes a main limitation: a review often spans several themes, and KMeans forces each one into a single cluster. That's why I moved to multi-label tagging with an LLM for the actual theme assignment, and kept clustering only for what it did well, deriving the taxonomy.

## Multi-label theme + sentiment tagging

Clustering surfaced the candidate taxonomy but showed that reviews are
multi-thematic. To respect that, I switched the assignment method: instead of
forcing each review into one cluster, I tag each review against the derived theme
list and allow **multiple themes per review**, each with its own sentiment.

I used a small, cheap classification model (Claude Haiku) for this. The taxonomy
was derived from the clustering above, not invented, so themes are grounded in
the data, then applied consistently at scale. 

In [ ]:
import os
import json
import time
import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-haiku-4-5"
IN_CSV = "data/combined_reviews.csv"
OUT_CSV = "data/tagged_reviews.csv"

THEMES = [
    "forced invite wall",
    "map and navigation",
    "ranking and rating mechanics",
    "data privacy and account",
    "performance and clunkiness",   # reloads, lag, bugs, UI friction
    "feature requests",             # categories, photo handling, editing
    "positive overall experience",
]

In [ ]:
SYSTEM = (
    "You are a precise product-feedback labeler. You assign app reviews to a fixed "
    "list of themes. A review may touch MORE THAN ONE theme, or none. For each theme "
    "the review actually touches, judge the sentiment expressed TOWARD THAT THEME."
)

def build_prompt(review_text):
    theme_block = "\n".join(f"- {t}" for t in THEMES)
    return (
        f"Themes:\n{theme_block}\n\n"
        f'Review:\n"""{review_text}"""\n\n'
        "Return ONLY a JSON object mapping each theme the review touches to a "
        'sentiment of "positive", "negative", or "neutral". Omit themes the review '
        'does not touch. If it touches none, return {}. No prose, no markdown, just JSON.'
    )

def tag_one(client, text):
    msg = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=SYSTEM,
        messages=[{"role": "user", "content": build_prompt(text)}],
    )
    raw = msg.content[0].text.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

In [ ]:
df = pd.read_csv(IN_CSV)
client = Anthropic()
print(f"{len(df)} reviews loaded")

for text in df["text"].astype(str).head(5):
    print(text[:100])
    print("  ->", tag_one(client, text), "\n")

In [ ]:
results = []
for i, text in enumerate(df["text"].astype(str)):
    results.append(tag_one(client, text))
    if (i + 1) % 25 == 0:
        print(f"tagged {i + 1}/{len(df)}")
    time.sleep(0.1)

df["theme_tags"] = [json.dumps(r, ensure_ascii=False) for r in results]

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(df)} tagged reviews -> {OUT_CSV}")

In [ ]:
tagged_reviews = pd.read_csv("data/tagged_reviews.csv")
tagged_reviews.head()